# polib

## Example 1: parse a catalog from text

In [ ]:
# Example 1: parse a catalog from text
import polib

po = polib.pofile(
    'msgid ""\nmsgstr ""\n"Language: de\\n"\n"Content-Type: text/plain; charset=UTF-8\\n"\n\n'
    'msgid "Hello"\nmsgstr "Hallo"\n\nmsgid "Goodbye"\nmsgstr ""\n'
)
assert isinstance(po, polib.POFile)
assert [entry.msgid for entry in po] == ["Hello", "Goodbye"]
assert po.find("Hello").msgstr == "Hallo"
assert po.metadata["Language"] == "de"
assert po.percent_translated() == 50, po.percent_translated()
print(f"{len(po)} entries, {po.percent_translated()}% translated, language {po.metadata['Language']}")

## Example 2: compile to `.mo` and read it with the standard library

In [ ]:
# Example 2: compile to .mo and read it with the standard library's gettext
import gettext
import pathlib
import tempfile

import polib

po = polib.POFile()
po.metadata = {"Content-Type": "text/plain; charset=UTF-8", "Language": "fr"}
po.append(polib.POEntry(msgid="Save", msgstr="Enregistrer"))
po.append(polib.POEntry(msgid="Open", msgstr="Ouvrir"))
mo_path = pathlib.Path(tempfile.mkdtemp()) / "app.mo"
po.save_as_mofile(str(mo_path))
with mo_path.open("rb") as handle:
    translations = gettext.GNUTranslations(handle)
assert translations.gettext("Save") == "Enregistrer"
assert translations.gettext("Open") == "Ouvrir"
assert translations.gettext("Missing") == "Missing"
print("gettext reads polib's .mo:", translations.gettext("Save"), "/", translations.gettext("Open"))

## Example 3: plural forms and fuzzy entries

In [ ]:
# Example 3: plural forms and fuzzy entries
import gettext
import io

import polib

po = polib.POFile()
po.metadata = {"Content-Type": "text/plain; charset=UTF-8", "Plural-Forms": "nplurals=2; plural=(n != 1);"}
po.append(polib.POEntry(msgid="%d file", msgid_plural="%d files", msgstr_plural={0: "%d Datei", 1: "%d Dateien"}))
po.append(polib.POEntry(msgid="Close", msgstr="Schliessen", flags=["fuzzy"]))
assert [entry.msgid for entry in po.fuzzy_entries()] == ["Close"]
assert [entry.msgid for entry in po.translated_entries()] == ["%d file"]
translations = gettext.GNUTranslations(io.BytesIO(po.to_binary()))
assert translations.ngettext("%d file", "%d files", 1) % 1 == "1 Datei"
assert translations.ngettext("%d file", "%d files", 3) % 3 == "3 Dateien"
# a fuzzy translation is a draft: it is not compiled, so the original text comes back
assert translations.gettext("Close") == "Close"
print("plural 1/3:", translations.ngettext("%d file", "%d files", 1) % 1, "/", translations.ngettext("%d file", "%d files", 3) % 3)

## Example 4: round trip through both formats

In [ ]:
# Example 4: .po text -> POFile -> .mo -> MOFile, nothing lost
import pathlib
import tempfile

import polib

po = polib.POFile()
po.metadata = {"Content-Type": "text/plain; charset=UTF-8"}
for number in range(20):
    po.append(polib.POEntry(msgid=f"message {number}", msgstr=f"Nachricht {number}", occurrences=[("app.py", str(number))]))
again = polib.pofile(str(po))
assert str(again) == str(po)
assert [(e.msgid, e.msgstr, e.occurrences) for e in again] == [(e.msgid, e.msgstr, e.occurrences) for e in po]
mo_path = pathlib.Path(tempfile.mkdtemp()) / "round.mo"
po.save_as_mofile(str(mo_path))
mo = polib.mofile(str(mo_path))
assert {e.msgid: e.msgstr for e in mo if e.msgid} == {e.msgid: e.msgstr for e in po}
print(f"{len(po)} entries survive .po and .mo round trips")

## Example 5: escapes, quotes and non-ASCII text

In [ ]:
# Example 5: escapes, quotes and non-ASCII text survive a save and a re-read
import pathlib
import tempfile

import polib

tricky = {"Line one\nLine two": "Zeile eins\nZeile zwei", 'Say "hi"': '„Hallo“ sagen', "Tab\there": "Tab\thier", "Café": "カフェ ☕"}
po = polib.POFile(wrapwidth=40)
po.metadata = {"Content-Type": "text/plain; charset=UTF-8"}
for msgid, msgstr in tricky.items():
    po.append(polib.POEntry(msgid=msgid, msgstr=msgstr))
path = pathlib.Path(tempfile.mkdtemp()) / "tricky.po"
po.save(str(path))
assert '\\n' in path.read_text(encoding="utf-8"), "newlines must be written escaped"
again = polib.pofile(str(path))
assert {e.msgid: e.msgstr for e in again} == tricky
print("escapes and non-ASCII text survive:", sorted(tricky)[:2])